# V2 · 踩坑版：当 `SequenceMatcher` 遇到“李世民”

> ⚠️ **这是一份“反面教材”，不要拿它上线。**
> 它的价值是让你亲眼看到：一个看起来很合理的“字面相似度对齐”，如何把整张图连成辐射状垃圾。

V2 在 V1 的基础上多做两件事：

1. 让模型多输出 `cause`（原因事件）和 `parent`（宏观事件）；
2. 用 `difflib.SequenceMatcher` 把 `cause` 和已有事件短语做字符相似度匹配，超过阈值就连 `CAUSED_BY`。

**已知后果**：模型把“李渊退位 / 李世民登基”这类最终事件当成很多事件的原因；
`SequenceMatcher` 又被“李世民”这个高频词欺骗，给出虚高分数。

In [ ]:
# ============================================================
# 0. 环境与路径：所有密钥都从 .env 读取，Notebook 里不写死
# ============================================================
import os
from pathlib import Path

from dotenv import load_dotenv

# Notebook 可能从 notebooks/ 启动，也可能从仓库根目录启动
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY", "").strip()
DEEPSEEK_MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-flash")
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")

# 默认不删库。第一次跑、或想从零复现时，才在 .env 里设 RESET_DB=true
RESET_DB = os.getenv("RESET_DB", "false").lower() == "true"

if not DEEPSEEK_API_KEY:
    raise RuntimeError("缺少 DEEPSEEK_API_KEY：请复制 .env.example 为 .env 并填写。")
if not NEO4J_PASSWORD:
    raise RuntimeError("缺少 NEO4J_PASSWORD：请复制 .env.example 为 .env 并填写。")

DATA_PATH = PROJECT_ROOT / "data" / "xuanwumen_text.txt"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"找不到文本文件：{DATA_PATH}")

print("仓库根目录:", PROJECT_ROOT)
print("文本文件:", DATA_PATH)
print("Neo4j:", NEO4J_URI, "| 用户:", NEO4J_USER)
print("RESET_DB:", RESET_DB)


In [ ]:
# ============================================================
# 2. 初始化 DeepSeek
# ============================================================
# 用官方 langchain-deepseek，而不是把 OpenAI 客户端硬指到 DeepSeek：
# 前者的字段、重试和流式行为与 DeepSeek API 更匹配。
from langchain_deepseek import ChatDeepSeek

llm = ChatDeepSeek(
    model=DEEPSEEK_MODEL,
    api_key=DEEPSEEK_API_KEY,
    temperature=0,  # 抽取任务尽量确定性
)
print("LLM 初始化完成:", DEEPSEEK_MODEL)


In [ ]:
# ============================================================
# 3. 读入文本
# ============================================================
text = DATA_PATH.read_text(encoding="utf-8").strip()
print(f"文本长度：{len(text)} 字符")


## 1. 让模型“顺手”推断 cause 和 parent

注意 Prompt 里已经要求“原因必须来自原文”。但**要求归要求**，模型在长文本里仍可能把
“登基”这种结局当作所有事件的原因。V2 保留这个原始设计，不做审核。

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """你是一个历史事件分析助手。请从文本中抽取事件，并为每个事件推断原因和所属宏观事件。

只输出一个 JSON 对象，不要输出 Markdown 代码块或解释：
{{"events": [
  {{"event": "事件短语", "time": "时间或 null", "participants": ["人名"],
    "cause": "导致该事件的前置事件，必须来自原文；没有则 null",
    "parent": "该事件所属的宏观事件；没有则 null"}}
]}}

文本：
{text}"""
)
print("Prompt 已就绪")


In [ ]:
import json

from langchain_core.output_parsers import JsonOutputParser

chain = prompt | llm | JsonOutputParser()
try:
    raw = chain.invoke({"text": text})
except Exception as exc:
    raise RuntimeError("模型没有返回合法 JSON，请检查 API Key / 模型名。") from exc

raw_events = raw.get("events", []) if isinstance(raw, dict) else []
if not isinstance(raw_events, list) or not raw_events:
    raise RuntimeError(f"抽取结果为空或格式异常：{raw!r}")


def normalize_events(items):
    cleaned = []
    for item in items:
        if not isinstance(item, dict):
            continue
        name = str(item.get("event") or "").strip()
        if not name:
            continue
        participants = item.get("participants") or []
        if not isinstance(participants, list):
            participants = [participants]
        cleaned.append({
            "event": name,
            "time": item.get("time") or None,
            "participants": sorted({str(p).strip() for p in participants if str(p).strip()}),
            "cause": str(item.get("cause") or "").strip() or None,
            "parent": str(item.get("parent") or "").strip() or None,
        })
    return cleaned


events = normalize_events(raw_events)
print(f"抽取到 {len(events)} 个事件（清洗后）")
for ev in events[:3]:
    print(json.dumps(ev, ensure_ascii=False))


In [ ]:
# ============================================================
# 3. 连接 Neo4j
# ============================================================
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD),
)
driver.verify_connectivity()
print("Neo4j 连接成功")


## 2. 先建 V1 同款底座

事件、人物、时序线与 V1 完全一致，只是 ID 前缀换成 `xuanwu_v2_`。

In [ ]:
def ensure_constraints(tx):
    tx.run("""
        CREATE CONSTRAINT event_id_unique IF NOT EXISTS
        FOR (e:Event) REQUIRE e.id IS UNIQUE
    """)
    tx.run("""
        CREATE CONSTRAINT person_name_unique IF NOT EXISTS
        FOR (p:Person) REQUIRE p.name IS UNIQUE
    """)


def reset_database(tx):
    tx.run("MATCH (n) DETACH DELETE n")


def build_v2_base(tx, ev_list):
    for order, ev in enumerate(ev_list):
        event_id = f"xuanwu_v2_{order:02d}"

        tx.run("""
            MERGE (e:Event {id: $id})
            SET e.name = $name,
                e.time = $time,
                e.order = $order,
                e.source = 'llm_extracted'
        """, id=event_id, name=ev["event"], time=ev["time"], order=order)

        for person in ev["participants"]:
            tx.run("""
                MERGE (p:Person {name: $name})
                WITH p
                MATCH (e:Event {id: $event_id})
                MERGE (p)-[r:PARTICIPATED_IN]->(e)
                SET r.source = 'llm_extracted'
            """, name=person, event_id=event_id)

        if order > 0:
            tx.run("""
                MATCH (prev:Event {id: $prev_id})
                MATCH (curr:Event {id: $curr_id})
                MERGE (prev)-[r:FOLLOWED_BY]->(curr)
                SET r.source = 'text_order'
            """, prev_id=f"xuanwu_v2_{order - 1:02d}", curr_id=event_id)


with driver.session() as session:
    session.execute_write(ensure_constraints)
    if RESET_DB:
        session.execute_write(reset_database)
        print("已清空数据库（RESET_DB=true）")
    session.execute_write(build_v2_base, events)

print("V2 底座建图完成")


## 3. 事故现场：`SequenceMatcher` 对齐 cause

这段代码的逻辑非常“朴素”：

- 对每个带 `cause` 的事件，遍历所有其他事件；
- 用字符级相似度找最像的那个；
- 分数 > 0.30 就建立 `(原因)-[:CAUSED_BY]->(结果)`。

它有三个致命问题：

1. **字面相似度不等于语义**：`cause` 往往是一长句话，事件是短句，分数天然偏低或虚高；
2. **高频词污染**：只要两边都有“李世民”，分数就会被抬高；
3. **阈值太低**：0.30 几乎等于“沾边就连”。

更隐蔽的问题是：如果 `cause` 恰好是“李世民登基”，模型会把**结局**当成**原因**。

In [ ]:
from difflib import SequenceMatcher


def similarity(a, b):
    """字符级相似度；仅用于复现问题，不要用于生产。"""
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, str(a), str(b)).ratio()


def build_v2_causal(tx, ev_list):
    accepted = []
    for order, ev in enumerate(ev_list):
        cause_text = ev.get("cause")
        if not cause_text:
            continue

        event_id = f"xuanwu_v2_{order:02d}"
        best_order, best_event, best_score = None, None, 0.0

        for other_order, other in enumerate(ev_list):
            if other_order == order:
                continue
            score = similarity(cause_text, other["event"])
            if score > best_score:
                best_order, best_event, best_score = other_order, other, score

        if best_event is None or best_score < 0.30:
            continue

        cause_id = f"xuanwu_v2_{best_order:02d}"
        tx.run("""
            MATCH (cause:Event {id: $cause_id})
            MATCH (effect:Event {id: $effect_id})
            MERGE (cause)-[r:CAUSED_BY]->(effect)
            SET r.source = 'sequence_matcher',
                r.score = $score,
                r.raw_cause = $raw_cause
        """, cause_id=cause_id, effect_id=event_id,
             score=round(best_score, 4), raw_cause=cause_text)

        accepted.append({
            "cause": best_event["event"],
            "effect": ev["event"],
            "score": round(best_score, 4),
            "time_inverted": best_order > order,
        })

    return accepted


with driver.session() as session:
    causal_edges = session.execute_write(build_v2_causal, events)

print(f"建立了 {len(causal_edges)} 条 CAUSED_BY")
for row in causal_edges[:12]:
    flag = "  <-- 时间倒置" if row["time_inverted"] else ""
    print(f"  {row['cause']}  ->  {row['effect']}  ({row['score']:.2f}){flag}")


## 4. 诊断：谁成了“万能原因”？

下面这条 Cypher 按出度排序。如果某个**结局事件**排在第一位，说明 V2 的幻觉已经发生。

In [ ]:
with driver.session() as session:
    top = session.run("""
        MATCH (cause:Event)-[r:CAUSED_BY]->(:Event)
        RETURN cause.name AS 被当成原因的事件,
               count(*) AS 出度,
               collect(r.score)[0..5] AS 部分分数
        ORDER BY 出度 DESC
        LIMIT 8
    """).data()

print("CAUSED_BY 出度 Top 8：")
for row in top:
    print(f"  {row['出度']:>2}  {row['被当成原因的事件']}  {row['部分分数']}")

print("\n如果第一名是‘李渊退位李世民登基’或类似结局，说明模型把终点当成了原因。")
print("这不是阈值调一调就能解决的问题，而是‘字面相似度’这条路本身不成立。")


## 5. 为什么翻车（以及为什么不在本 Notebook 里修）

**根因不是某一个参数，而是三个假设同时成立：**

1. 假设模型会把 `cause` 写成“原文中的另一个事件短语”——实际上它经常写成一句长描述；
2. 假设“字面最像”等于“语义上真的是原因”——高频人名会让这个假设失效；
3. 假设“最终事件不会成为原因”——但模型没有时序概念，它只知道“登基”很重要。

**还有一个工程细节**：原始原型里 `other` 变量在 `for` 循环结束后仍然指向最后一个事件，
日志里打印的名字可能和真正写入的 `best_match_id` 不一致。这类“变量泄漏”会让排错方向完全跑偏。

V3 不再用“字面相似度找原因”，而是改成：

- 让模型输出**完整事件短语**，代码做精确匹配；
- 精确匹配失败才做高阈值模糊匹配（0.86），且 Top-2 分数接近时直接丢弃；
- 写入前强制检查 `cause.order < effect.order`；
- 审核阶段带入原文，明确“终点禁律”。

> 换句话说：**LLM 负责提议，代码负责裁决。**

In [ ]:
driver.close()
print("Neo4j 连接已关闭")
